In [ ]:
import math

class Value : 
    def __init__(self, data, _children = (), _op = '', label = '') :
        self.data = data
        self._prev = set(_children)
        self.grad = 0.0
        self._backward = lambda : None
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"data({self.data})"

    def __add__(self, other) :
        other = Value(other) if isinstance(other, (int, float)) else other 
        out = Value(self.data + other.data, _children = (self, other), _op = '+')

        def _backward() : 
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
    
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __mul__(self, other) :
        other = Value(other) if isinstance(other, (int, float)) else other 
        out = Value(self.data * other.data, _children = (self, other),  _op = '*')

        def _backward() : 
            self.grad += (other.data)*out.grad
            other.grad += (self.data)*out.grad

        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)) 
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward() : 
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self*other**-1

    def __neg__(self):
        return self * -1
    
    def tanh(self) : 
        n = self.data
        t = (math.exp(2*n)- 1)/(math.exp(2*n)+1)
        out = Value(t,(self,), 'tanh')

        def _backward() : 
            self.grad += (1 - t**2)*out.grad
        out._backward = _backward 
        return out

    def backward(self) : 
        topo = []
        visited = set()
        def build_topo(v) : 
            if v not in visited : 
                visited.add(v)
                for child in v._prev : 
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1 
        for child in reversed(topo) : 
            child._backward()







In [22]:
a = Value(-4.0)
b = Value(2.0)
c = a + b
d = a * b + b**3
c += c + 1
d += d * 2 + (b + a).tanh()
e = c + d
f = e**2
g = f / 2.0
g.backward()

In [ ]:
import random


class Neuron :
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for i in range(nin)]
        self.b = Value(0)


    def __call__(self, x) : 
        act = sum([wi*xi for (wi, xi) in zip(self.w, x)]) + self.b
        out = act.tanh()
        return out

    def parameters(self) : 
        return self.w + [self.b]



class Layers : 
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for i in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self) : 
        return [p for n in self.neurons for p in n.parameters()]


class MLP : 

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layers(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers : 
            x = layer(x)

        return x[0] if len(x) == 1 else x

    def parameters(self) : 
        return [p for l in self.layers for p in l.parameters()]
        



